# Task 2 — Data Integration and Cleaning

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

This notebook cleans the Transport for NSW charger dataset acquired in Task 1 and integrates it
with ABS SA4 boundaries. It addresses missing values, inconsistent text and types, duplicate
records, conflicting charger attributes and spatial-reference differences. Every transformation is
recorded in a cleaning ledger, and uncertain values are flagged rather than replaced without
supporting evidence.

### How to run

Run `01_data_acquisition.ipynb` first. This notebook locates its inputs through
`data/raw/manifest.json` and stops with a clear message if a required file is missing. Then run the
cells from top to bottom.

### Contents

1. [Approach](#t2-approach)
2. [Setup](#t2-setup)
3. [Structural cleaning](#t2-structure)
4. [Operator names](#t2-operators)
5. [Charger type and status](#t2-type)
6. [Charger power ratings](#t2-rating)
7. [Addresses and postcodes](#t2-address)
8. [Coordinates and sites](#t2-coords)
9. [Duplicates and reconciliation](#t2-dupes)
10. [Derived rating fields](#t2-ratingfields)
11. [Cross-field consistency](#t2-consistency)
12. [Spatial integration with ASGS SA4](#t2-spatial)
13. [Validation and summary](#t2-validate)
14. [Outputs](#t2-outputs)

<a id="t2-approach"></a>
## 1. Approach

The notebook converts the profiled source data into a consistent, traceable dataset for downstream
augmentation and storage. Three principles guide the process for data cleaning:

**Record every change.** Each transformation appends its name, purpose and affected-record count to
`data/interim/cleaning_log.json`. This provides auditable evidence of what changed and supports
quantified statements such as “no exact duplicates were found and 12 conflicting records were
reconciled under stated rules.”

**Repair only with a defensible rule.** Mechanical problems such as repeated whitespace can be
corrected safely. Ambiguous values, such as a placeholder locality or an unresolved truncated
operator, are retained and marked with a boolean quality flag rather than replaced with a guess.

**Apply transformations in a deliberate order.** Text is normalised before comparison, operator
names are canonicalised before duplicate detection, and derived power fields are calculated after
duplicate reconciliation so they describe the retained record.

### Outputs

| File | Contents |
|---|---|
| `data/interim/ev_chargers_clean.csv` | one cleaned charger record per row, including SA4 fields and quality flags |
| `data/interim/charger_power_ratings.csv` | parsed power-rating components linked to charger records |
| `data/interim/probable_duplicates.csv` | evidence for records that may describe the same charger across different feeds |
| `data/interim/sa4_regions_nsw.csv` | NSW SA4 reference data, including boundary-availability information |
| `data/interim/cleaning_log.json` | each cleaning operation and its affected-record count |

<a id="t2-setup"></a>
## 2. Setup



In [1]:
%pip install -q "pandas>=2.2" "numpy>=1.26" "duckdb>=1.5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

# --- Locate the Task 1 outputs ---------------------------------------------
# Paths come from the manifest Task 1 wrote, so Task 2 depends on Task 1's
# outputs rather than on its variables still being in memory.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
MANIFEST_PATH = RAW_DIR / "manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "data/raw/manifest.json is missing - run 01_data_acquisition.ipynb first."
    )
_artefacts = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))["artefacts"]
EV_CSV_PATH = PROJECT_ROOT / _artefacts["tfnsw_ev_charging_locations"]["path"]
SA4_SHAPEFILE_PATH = PROJECT_ROOT / _artefacts["abs_asgs_sa4_boundaries"]["shapefile"]
for required in (EV_CSV_PATH, SA4_SHAPEFILE_PATH):
    if not required.exists():
        raise FileNotFoundError(
            f"{required.relative_to(PROJECT_ROOT)} is missing - run 01_data_acquisition.ipynb first."
        )
for directory in (INTERIM_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Task 2 outputs ---------------------------------------------------------
CLEAN_CHARGERS_PATH = INTERIM_DIR / "ev_chargers_clean.csv"
RATINGS_PATH = INTERIM_DIR / "charger_power_ratings.csv"
DUPLICATES_PATH = INTERIM_DIR / "probable_duplicates.csv"
SA4_NSW_PATH = INTERIM_DIR / "sa4_regions_nsw.csv"
CLEANING_LOG_PATH = INTERIM_DIR / "cleaning_log.json"

# --- Reference values used by the cleaning rules ----------------------------
NSW_LAT_RANGE = (-37.6, -28.1)
NSW_LON_RANGE = (140.9, 153.7)
COORDINATE_DECIMALS = 6          # ~0.11 m - finer than the source can justify
NSW_POSTCODE_RANGES = ((1000, 1999), (2000, 2599), (2619, 2899), (2921, 2999))

SOURCE_CRS = "EPSG:7844"         # GDA2020, the datum of the ABS boundaries
TARGET_CRS = "EPSG:4326"         # WGS84, the datum of the TfNSW lat/lon columns

# `cleaning_log` accumulates one entry per transformation: what was done and how
# many records it touched. It is written to disk at the end, so every change the
# cleaning made can be checked and quantified.
cleaning_log = []


def record_step(step: str, detail: str, affected: int) -> None:
    """Append one auditable entry to the cleaning ledger and echo it."""
    cleaning_log.append({"step": step, "detail": detail, "records_affected": int(affected)})
    print(f"  {step:<26} {affected:>6,}  {detail}")


print(f"EV CSV    : {EV_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shapefile : {SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)}")

EV CSV    : data\raw\tfnsw\ev_20251216.csv
Shapefile : data\raw\abs\sa4_shapefile\SA4_2026_AUST_GDA2020.shp


### 3.1 Read the source file as text

Default type inference is useful for profiling but can alter values before a cleaning decision is
made. The source CSV is therefore read with every column as text. Each conversion is then applied
explicitly in a later step, where its rule, order and effect can be inspected and counted.

In [3]:
# `dtype=str` for every column, deliberately. pandas' type inference is the first
# place data gets silently altered: PCODE would become an integer (losing any
# leading zero and turning the 121 missing values into floats) and OBJECTID would
# become a float for the same reason. Reading everything as text means every
# conversion below is explicit, ordered, and reviewable.
raw = pd.read_csv(EV_CSV_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)

print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns, all as text")
raw.head(3)

Loaded 1,958 rows x 12 columns, all as text


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,,,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.81100405,150.8495966,Blacktown City Council,2766,Existing Fast Chargers
2,,,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.5118739,151.669395,Central Darling Shire Council,2350,TfNSW Regional


### 3.2 Normalise whitespace and missing values

Whitespace is cleaned first because subsequent comparisons depend on consistent text. The source
contains 733 addresses with embedded newlines and two operator values with trailing spaces. These
differences would otherwise prevent equivalent records and operator names from matching.

Repeated whitespace is collapsed, leading and trailing spaces are removed, and empty strings are
converted to proper missing values. This gives `isna()`, grouping operations and database
constraints one consistent representation of missing data.

In [4]:
def squash_whitespace(value):
    """
    Collapse newlines, tabs and runs of spaces into single spaces, then trim.

    This runs before everything else because every later comparison - dedup,
    operator matching, address parsing - compares strings. 733 addresses contain
    embedded newlines and two operator names carry a trailing space, so without
    this step a large share of the dataset fails to match its own duplicate.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if text else pd.NA


df = raw.copy()
text_columns = df.columns.tolist()

blank_cells = int((df[text_columns] == "").sum().sum())
for column in text_columns:
    df[column] = df[column].map(squash_whitespace)
rewritten = int((raw[text_columns].astype(str) != df[text_columns].astype(str)).sum().sum())

print("Normalising whitespace and converting empty strings to NA")
record_step("whitespace normalised", "cells whose text was rewritten", rewritten - blank_cells)
record_step("empty -> NA", "empty strings converted to missing values", blank_cells)

Normalising whitespace and converting empty strings to NA
  whitespace normalised         808  cells whose text was rewritten
  empty -> NA                 3,638  empty strings converted to missing values


### 3.3 Standardise column names and data types

The source combines several naming conventions, including `OBJECTID`, `Station_name` and
`LGANAME`. Columns are renamed to descriptive `snake_case` names so the cleaned file and SQL schema
use consistent identifiers.

Numeric conversion uses `errors="coerce"`, which turns an unparseable value into missing data. Each
conversion therefore records how many new missing values it creates. Nullable pandas `Int64` is
used for identifiers and counts because standard NumPy integers cannot represent missing values;
coordinates are stored as floating point and rounded to six decimal places.

In [5]:
# snake_case throughout. The source mixes three conventions (`OBJECTID`,
# `Station_name`, `LGANAME`), and the destination is a SQL schema in Task 4 where
# one convention avoids quoted identifiers everywhere.
COLUMN_RENAMES = {
    "OBJECTID": "source_object_id",
    "Station_name": "station_name",
    "Station_address": "station_address_raw",
    "Operator": "operator_raw",
    "Number_of_plugs": "number_of_plugs",
    "Charger_Type": "charger_type_raw",
    "Charger_rating": "charger_rating_raw",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "LGANAME": "lga_name",
    "PCODE": "postcode_reported",
    "Source": "source_feed",
}
df = df.rename(columns=COLUMN_RENAMES)

# `errors="coerce"` turns anything unparsable into NA rather than raising - and
# the number of values it silently created is logged, so a bad coercion can never
# slip past unnoticed. Int64 (capital I) is pandas' nullable integer: plain int64
# cannot hold a missing value and would force these columns back to float.
numeric_specs = {
    "source_object_id": "Int64",
    "number_of_plugs": "Int64",
    "latitude": "float64",
    "longitude": "float64",
}
for column, dtype in numeric_specs.items():
    before_missing = df[column].isna().sum()
    converted = pd.to_numeric(df[column], errors="coerce")
    df[column] = converted.astype(dtype) if dtype == "Int64" else converted
    coerced = int(df[column].isna().sum() - before_missing)
    if coerced:
        record_step("type coercion", f"{column}: unparsable values set to NA", coerced)

# Coordinates rounded to 6 dp. The source carries more decimals than a GPS fix
# justifies, and unrounded floats make two records at one physical site compare
# as different locations.
df["latitude"] = df["latitude"].round(COORDINATE_DECIMALS)
df["longitude"] = df["longitude"].round(COORDINATE_DECIMALS)

df.dtypes.to_frame("dtype")

,dtype
source_object_id,Int64
station_name,object
station_address_raw,object
operator_raw,object
number_of_plugs,Int64
charger_type_raw,object
charger_rating_raw,object
latitude,float64
longitude,float64
lga_name,object


<a id="t2-operators"></a>
## 4. Standardise operator names

The source contains 50 operator strings with case differences, spacing variants, aliases and
truncations. Mechanical variations such as `ChargeHub` and `Charge Hub` can be standardised
directly, while pairs such as `BP` and `BP Australia` require an explicit known alias.

Mappings are kept in a reviewable lookup table rather than inferred through fuzzy similarity. This
prevents unrelated names such as `Charge Hub` and `Charge OS` from being merged simply because they
share similar text. Full forms that appear elsewhere in the release support mappings such as
`Viva Energy A` to `Viva Energy Australia`; unresolved values such as `University of`,
`Energy Austra` and `Fast Cities A` remain unchanged and are flagged. Operators absent from the
lookup are displayed so new values in a future release can be reviewed deliberately.

In [6]:
# Three problems live in `Operator`, and only the first can be fixed by a rule:
# case/punctuation variants, trailing whitespace, and names truncated at 13-14
# characters by a fixed-width field upstream. Truncation is irreversible from the
# data alone, so the repairs are stated in a lookup table a human can review -
# not inferred by a string-similarity heuristic, which would also merge the
# unrelated 'Charge Hub' and 'Charge OS'.
OPERATOR_CANONICAL = {
    # case and spacing variants
    "chargehub": "ChargeHub",
    "charge hub": "ChargeHub",
    "non-networked": "Non-networked",
    "non networked": "Non-networked",
    # short form and full name of one company
    "bp": "BP Australia",
    "bp australia": "BP Australia",
    "tesla": "Tesla",
    "tesla motors": "Tesla",
    "nrma": "NRMA",
    "nrma electric": "NRMA",
    "evie": "Evie Networks",
    "evie networks": "Evie Networks",
    # cut off mid-word by the upstream fixed-width field
    "plus es": "PLUS ES",
    "plus es manag": "PLUS ES",
    "viva energy a": "Viva Energy Australia",
    "viva energy australia": "Viva Energy Australia",
}

# Deliberately NOT merged:
#   'Charge Hub' vs 'Charge OS'                          unrelated companies
#   'Porsche Destination Charging' vs
#   'Porsche Smart Mobility'                             two distinct programmes
# Truncations that cannot be resolved without an external source are flagged
# rather than guessed at. 'Viva Energy A' and 'PLUS ES Manag' are repaired
# above because their full forms also appear in this release; these three
# have no full form anywhere in it:
UNRESOLVED_TRUNCATIONS = {"university of", "energy austra", "fast cities a"}


def canonical_operator(value):
    """Map a raw operator string to its canonical form via the lookup table."""
    if pd.isna(value):
        return pd.NA
    key = re.sub(r"\s+", " ", str(value)).strip().lower()
    return OPERATOR_CANONICAL.get(key, str(value).strip())


operator_key = df["operator_raw"].fillna("").str.strip().str.lower()
df["operator"] = df["operator_raw"].map(canonical_operator)
df["operator_truncated_flag"] = operator_key.isin(UNRESOLVED_TRUNCATIONS)

print(f"Operator values: {df['operator_raw'].nunique()} raw -> {df['operator'].nunique()} canonical")
record_step("operator canonicalised", "values rewritten to a canonical name",
            int((df["operator"] != df["operator_raw"]).sum()))
record_step("operator truncated", "unresolved truncations flagged for review",
            int(df["operator_truncated_flag"].sum()))

# Anything absent from the lookup table is listed, so the table can be extended
# deliberately when TfNSW publishes a release containing new operators.
unmapped = sorted(set(df.loc[~operator_key.isin(OPERATOR_CANONICAL), "operator"].dropna()))
print(f"\n{len(unmapped)} operator(s) passed through unmapped (whitespace-normalised only):")
print("   ", ", ".join(unmapped))

Operator values: 50 raw -> 42 canonical
  operator canonicalised        181  values rewritten to a canonical name
  operator truncated             19  unresolved truncations flagged for review

34 operator(s) passed through unmapped (whitespace-normalised only):
    360 EV Charge, AXCharge, Alchemy Charge, Ampol, BMW, CasaCharge, Charge OS, ChargePoint, ChargePost, Chargefox, Chargestar, Counties Energy, EV Meter, EVE Australia, EVNet, EVSE, EVUp, EVX, Elanga, Energy Austra, Engie, Everty, Exploren, Fast Cities A, Gentari, JOLT, Noodoe, Porsche Destination Charging, Porsche Smart Mobility, Saascharge, Smart Charge, University of, Wevolt, Zeus Renewables


<a id="t2-type"></a>
## 5. Separate charger type and status

`Charger_Type` stores two different concepts: the electrical type (`AC` or `DC`) and, for 98
records, the lifecycle status `Upcoming`. Keeping both concepts in one field makes type and status
queries ambiguous.

The field is separated into `charger_type` and `charger_status`. AC and DC records are marked as
operational. Upcoming records retain that status, while their electrical type remains missing
because the source does not state whether the planned installation will be AC or DC. A high power
rating may suggest DC, but it is not used to replace an unreported value.

In [7]:
# `Charger_Type` holds two different facts: the electrical type (AC/DC) and, for
# 98 records, a lifecycle status ('Upcoming'). Keeping them in one column means
# no query can ask for "all DC chargers" without silently excluding planned DC
# sites, so the two facts are separated into two fields.
def split_type_status(value):
    if pd.isna(value):
        return (pd.NA, pd.NA)
    text = str(value).strip().upper()
    if text in {"AC", "DC"}:
        return (text, "Operational")
    if text == "UPCOMING":
        # The source does not say whether a planned site will be AC or DC, so the
        # type is left missing rather than guessed from the power rating.
        return (pd.NA, "Upcoming")
    return (pd.NA, "Unknown")


df[["charger_type", "charger_status"]] = pd.DataFrame(
    df["charger_type_raw"].map(split_type_status).tolist(), index=df.index
)

record_step("type/status separated", "'Upcoming' moved into charger_status",
            int((df["charger_status"] == "Upcoming").sum()))
df.groupby(["charger_status", "charger_type"], dropna=False).size().to_frame("records")

  type/status separated          98  'Upcoming' moved into charger_status


records
charger_status charger_type         
Operational    AC               1427
               DC                433
Upcoming       NaN                98

<a id="t2-rating"></a>
## 6. Parse charger power ratings

`Charger_rating` contains 46 distinct strings across four formats: values with units (`22 kW`),
bare numbers (`22`), the non-numeric placeholder `AC`, and compound descriptions such as
`2x350kW & 2x175kW`.

Compound ratings require separate treatment because they describe several connector groups in one
text value. The parser returns a list of `(connector_count, power_kw)` pairs, preserving both the
350 kW and 175 kW components instead of reducing the rating to one maximum or average. The `AC`
placeholder is treated as a missing power value rather than zero because it describes current type,
not power.

This section defines and profiles the parser. Derived fields are calculated after duplicate
reconciliation so they describe the final retained records.

In [8]:
RATING_SEPARATOR = re.compile(r"\s*(?:&|\+|,|/|\band\b)\s*", re.IGNORECASE)
RATING_TOKEN = re.compile(
    r"^(?:(?P<count>\d+)\s*x\s*)?(?P<kw>\d+(?:\.\d+)?)\s*(?:kw)?$", re.IGNORECASE
)


def parse_rating(value):
    """
    Turn one raw `Charger_rating` string into a list of (connectors, kW) pairs.

    Covers all four formats present in the source:
        '22 kW'              -> [(1, 22.0)]
        '22'                 -> [(1, 22.0)]
        '2x350kW & 2x175kW'  -> [(2, 350.0), (2, 175.0)]
        'AC'                 -> []            a placeholder, not a power value

    Returning a list rather than one number is the whole point: a compound value
    describes several connectors of different power at one site, and flattening
    it to a single figure destroys information the Task 4 schema needs.
    """
    if pd.isna(value):
        return []
    text = str(value).strip().lower().replace("\u00d7", "x")
    if not text:
        return []
    components = []
    for part in RATING_SEPARATOR.split(text):
        match = RATING_TOKEN.match(part.strip())
        if match:
            count = int(match.group("count")) if match.group("count") else 1
            components.append((count, float(match.group("kw"))))
    return components


def classify_rating(value) -> str:
    """Label which of the four formats a raw rating string uses."""
    text = "" if pd.isna(value) else str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?\s*kW", text, re.IGNORECASE):
        return "number+unit"
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return "bare number"
    if re.search(r"\dx\s*\d", text, re.IGNORECASE):
        return "compound"
    if re.fullmatch(r"[A-Za-z ]+", text):
        return "non-numeric placeholder"
    return "other"


# Profiling only at this point. The derived numeric columns are computed after
# deduplication, so they are guaranteed to describe the record that survives.
df["charger_rating_raw"].map(classify_rating).value_counts().to_frame("records")

,records
charger_rating_raw,
number+unit,1315
non-numeric placeholder,522
compound,99
bare number,22


<a id="t2-address"></a>
## 7. Normalise addresses and postcodes

After whitespace cleaning, address punctuation is standardised by removing repeated commas,
correcting spaces around separators and trimming leading or trailing punctuation. This produces a
consistent single-line address without attempting to infer missing street or locality text.

The source contains both an address postcode and a separate `PCODE` field. `PCODE` is missing in
121 raw records and is incorrect in some others; for example, a Wilcannia record reports 2350 while
its address and coordinates indicate 2836. The postcode extracted from the address is therefore
preferred, with `PCODE` used only as a fallback. Disagreements and values outside configured NSW
ranges are retained as flags.

The locality is not inferred. In 399 records, `Sydney` is used as a placeholder for another suburb.
These records are flagged, while the later SA4 spatial join supplies a reliable regional attribute.

In [9]:
POSTCODE_IN_ADDRESS = re.compile(r"\b(\d{4})\b(?!.*\b\d{4}\b)")   # last 4-digit run


def in_nsw_postcode_range(code) -> bool:
    if pd.isna(code) or not str(code).isdigit():
        return False
    number = int(code)
    return any(low <= number <= high for low, high in NSW_POSTCODE_RANGES)


# The newlines are already gone (whitespace pass); what remains is punctuation
# spacing - ' ,', ',,' and stray leading/trailing commas from empty street lines.
df["station_address"] = (
    df["station_address_raw"]
    .str.replace(r"\s*,\s*", ", ", regex=True)
    .str.replace(r"(?:,\s*){2,}", ", ", regex=True)
    .str.replace(r"^[,\s]+", "", regex=True)
    .str.replace(r"[,\s]+$", "", regex=True)
)

df["postcode_from_address"] = df["station_address"].str.extract(POSTCODE_IN_ADDRESS, expand=False)
# The address is the more trustworthy of the two sources: PCODE is null in 121
# records and demonstrably wrong in others (the Wilcannia record carries 2350 but
# its address, and its coordinates, are in 2836).
df["postcode"] = df["postcode_from_address"].fillna(df["postcode_reported"])
df["postcode_conflict_flag"] = (
    df["postcode_from_address"].notna()
    & df["postcode_reported"].notna()
    & (df["postcode_from_address"] != df["postcode_reported"])
)
df["postcode_valid_flag"] = df["postcode"].map(in_nsw_postcode_range)

# The locality inside the address is a known-bad field: 399 records use 'Sydney'
# in place of the real suburb. It is flagged, not repaired - the SA4 join below
# supplies a trustworthy region instead.
df["locality_placeholder_flag"] = df["station_address"].str.contains(
    r",\s*Sydney\s*,", case=False, regex=True, na=False
)

record_step("addresses normalised", "addresses reduced to one clean line",
            int(df["station_address"].notna().sum()))
record_step("postcode recovered", "postcode recovered from the address text",
            int((df["postcode_reported"].isna() & df["postcode"].notna()).sum()))
record_step("postcode conflict", "PCODE disagrees with the address postcode",
            int(df["postcode_conflict_flag"].sum()))
record_step("postcode invalid", "postcode outside the NSW ranges",
            int((~df["postcode_valid_flag"]).sum()))
record_step("locality placeholder", "'Sydney' used in place of the real suburb",
            int(df["locality_placeholder_flag"].sum()))
df[["station_address", "postcode_reported", "postcode", "postcode_conflict_flag"]].head()

  addresses normalised        1,958  addresses reduced to one clean line
  postcode recovered            120  postcode recovered from the address text
  postcode conflict              35  PCODE disagrees with the address postcode
  postcode invalid                3  postcode outside the NSW ranges
  locality placeholder          399  'Sydney' used in place of the real suburb


,station_address,postcode_reported,postcode,postcode_conflict_flag
0,"Muswellbrook, 2333",2333,2333,False
1,"01 Wallgrove Road, Sydney, 2766",2766,2766,False
2,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",2350,2836,True
3,"1 Balfour St, Sydney, 2070",2070,2070,False
4,"1 Bay Ln, Byron Bay, 2481",2481,2481,False


<a id="t2-coords"></a>
## 8. Validate coordinates and identify sites

Coordinates are rounded to six decimal places, which is already finer than the source accuracy can
justify. Rounding also prevents insignificant floating-point differences from creating separate
locations.

`site_id` groups records with the same rounded coordinates. Shared coordinates do not automatically
indicate duplication: one physical location can contain separate AC and DC installations or
equipment operated by different networks. Separating the concepts of site and charger allows the
duplicate rules to reconcile repeated descriptions without removing legitimate co-located records.

In [10]:
missing_coords = df["latitude"].isna() | df["longitude"].isna()
outside_nsw = ~(
    df["latitude"].between(*NSW_LAT_RANGE) & df["longitude"].between(*NSW_LON_RANGE)
)
df["coordinate_valid_flag"] = ~(missing_coords | outside_nsw)

record_step("coordinates missing", "records with no usable coordinates", int(missing_coords.sum()))
record_step("coordinates off-NSW", "records outside the NSW bounding box",
            int((outside_nsw & ~missing_coords).sum()))

# `site_id` groups records that share a physical location. It is what makes the
# difference between a duplicate (same site, same charger, two feeds) and a
# legitimate co-located pair (same site, one AC unit and one DC unit) something
# the data can express, instead of something dedup has to guess at.
site_key = (
    df["latitude"].round(COORDINATE_DECIMALS).astype("string")
    + "," + df["longitude"].round(COORDINATE_DECIMALS).astype("string")
)
df["site_id"] = pd.factorize(site_key)[0] + 1
print(f"{df['site_id'].nunique():,} distinct sites across {len(df):,} records")

  coordinates missing             0  records with no usable coordinates
  coordinates off-NSW             0  records outside the NSW bounding box
1,935 distinct sites across 1,958 records


<a id="t2-dupes"></a>
## 9. Reconcile duplicate records

Duplicate handling uses two passes for records at the same site, followed by a cross-feed review in
Section 9.1.

**Exact duplicates** agree on every field used to identify a charger. Additional copies contain no
new information and can be removed.

**Conflicting duplicates** share a site, operator, charger type and status but disagree in other
fields. Removing either row could discard useful information, so each group is reconciled using
field-specific rules:

| Field | Rule | Rationale |
|---|---|---|
| rating | prefer a parsable value, then the value describing more components | retains numeric and compound detail over a placeholder or simpler value |
| plugs | retain the larger reported count | preserves the more complete reported installation count |
| text fields | retain the longest non-null value | reduces loss from truncated descriptions |
| flags | logical OR | preserves a warning raised by either source row |

The grouping key is `(site_id, operator, charger_type, charger_status)`. Operator standardisation
must occur first so aliases such as `Tesla` and `Tesla Motors` do not prevent equivalent records
from being grouped.

`charger_id` is assigned after reconciliation, when the record set is stable. It becomes the key
used by subsequent augmentation and database tables.

In [11]:
# Two different problems, handled in two passes.
#
# Pass 1 - exact duplicates: identical on every field that identifies a charger.
# One copy is simply redundant.
IDENTITY_COLUMNS = [
    "site_id", "operator", "charger_type", "charger_status",
    "charger_rating_raw", "number_of_plugs", "station_address",
]
exact_duplicates = df.duplicated(subset=IDENTITY_COLUMNS, keep="first")
df = df.loc[~exact_duplicates].copy()
record_step("exact duplicates", "identical records removed", int(exact_duplicates.sum()))

# Pass 2 - conflicting duplicates: the same charger at the same site described
# differently by two feeds (rating 'AC' with 4 plugs in one row, '7' with 2 plugs
# in the other). Dropping either row loses real information, so the group is
# merged under rules chosen per field:
#   rating  - prefer the value that actually parses to a power figure, then the
#             one describing the most connectors; the 'AC' placeholder loses
#   plugs   - the larger count; a feed reporting fewer is reporting a subset
#   text    - the longest non-null value, i.e. the least truncated
#   flags   - OR'd, so a problem noted on either row survives the merge
GROUP_KEY = ["site_id", "operator", "charger_type", "charger_status"]
conflicting = df.duplicated(subset=GROUP_KEY, keep=False)
conflict_groups = int(df.loc[conflicting, GROUP_KEY].drop_duplicates().shape[0])
print(f"{int(conflicting.sum())} record(s) in {conflict_groups} conflicting group(s)")


def longest(series):
    """The longest non-null string in the group - the least truncated one."""
    values = series.dropna().astype(str)
    return max(values, key=len) if len(values) else pd.NA


def best_rating(series):
    """
    The most informative rating in the group.

    Sort key: parses at all > describes more connectors > longer string. This is
    what keeps '7' rather than the placeholder 'AC', and keeps
    '2x350kW & 2x175kW' rather than the single-figure '175 kW'.
    """
    values = series.dropna().astype(str)
    if not len(values):
        return pd.NA
    return max(values, key=lambda v: (len(parse_rating(v)) > 0, len(parse_rating(v)), len(v)))


AGGREGATIONS = {
    "source_object_id": "first",
    "station_name": longest,
    "station_address": longest,
    "station_address_raw": longest,
    "operator_raw": longest,
    "charger_type_raw": "first",
    "charger_rating_raw": best_rating,
    "number_of_plugs": "max",
    "latitude": "first",
    "longitude": "first",
    "lga_name": longest,
    "postcode": longest,
    "postcode_reported": longest,
    "postcode_from_address": longest,
    "source_feed": longest,
    "operator_truncated_flag": "max",
    "postcode_conflict_flag": "max",
    "postcode_valid_flag": "max",
    "locality_placeholder_flag": "max",
    "coordinate_valid_flag": "min",
}

before_rows = len(df)
df = df.groupby(GROUP_KEY, dropna=False, as_index=False).agg(AGGREGATIONS)
record_step("duplicates reconciled", "conflicting records merged into one",
            before_rows - len(df))

# A stable surrogate key, assigned once the record set is final. Task 4 uses it
# as the primary key; Task 3 uses it as the join target for augmented attributes.
df = df.sort_values(["site_id", "operator", "charger_type"]).reset_index(drop=True)
df.insert(0, "charger_id", range(1, len(df) + 1))
print(f"{before_rows:,} -> {len(df):,} records after reconciliation")

  exact duplicates                0  identical records removed
24 record(s) in 12 conflicting group(s)
  duplicates reconciled          12  conflicting records merged into one
1,958 -> 1,946 records after reconciliation


### 9.1 Identify probable duplicates across feeds

The preceding rules compare records sharing the same rounded coordinates. The same charger can also
appear in different programme feeds with coordinates a few metres apart or with differently
formatted versions of the same address.

A cross-feed pair is flagged as a probable duplicate when operator, charger type and status agree,
the source feeds differ, and either:

- the coordinates are within 2 m; or
- the normalised street number and street name agree and the coordinates are within 50 m.

The rule is deliberately conservative. Records are flagged rather than merged because proximity
and address agreement do not prove that two rows represent one physical charger. Same-feed pairs
are also left unchanged because two units at one address may be separate installations.

The suspected record receives `probable_duplicate_flag = True`, and the pairwise evidence is saved
to `probable_duplicates.csv`. This preserves the information required for review without deleting a
potentially distinct charger.

In [12]:
# Pass 3 - the same charger in two source feeds. Passes 1 and 2 compare only
# records at one site (identical coordinates), so a charger that two feeds
# report a fraction of a metre apart survives both. Pairs are flagged, not
# merged: the evidence is strong but not conclusive, and merging would
# renumber charger_id, which Task 3's cached API searches are keyed on.
SAME_POINT_M = 2              # within GPS precision: one coordinate written twice
SAME_ADDRESS_RADIUS_M = 50    # a shared street address counts only this close

STREET_SUFFIXES = {
    "street": "st", "road": "rd", "avenue": "ave", "highway": "hwy", "drive": "dr",
    "parade": "pde", "lane": "ln", "place": "pl", "crescent": "cres", "court": "ct",
    "terrace": "tce", "boulevard": "blvd", "circuit": "cct", "close": "cl",
    "esplanade": "esp", "square": "sq", "grove": "gr",
}
STREET_ADDRESS = re.compile(
    r"^(?P<number>\d+[a-z]?(?:-\d+[a-z]?)?) (?P<street>[a-z ]+? (?:"
    + "|".join(sorted(set(STREET_SUFFIXES.values()) | {"way", "row", "mews"}))
    + r"))\b"
)


def street_key(address):
    """
    Reduce an address to 'number street', e.g. '106 parraween st', or None.

    Only the house number and street are compared, because the locality is the
    unreliable part of these addresses (the 'Sydney' placeholder, section 7).
    An address with no house number gives no key: 'Gardeners Rd' alone is too
    vague to say that two records describe the same charger.
    """
    if pd.isna(address):
        return None
    text = str(address).lower().replace("/", "-")
    text = re.sub(r"\s*-\s*", "-", text)                 # '2 - 8' -> '2-8'
    text = re.sub(r"[^a-z0-9\- ]", " ", text)
    for long_form, short_form in STREET_SUFFIXES.items():
        text = re.sub(rf"\b{long_form}\b", short_form, text)
    match = STREET_ADDRESS.match(re.sub(r"\s+", " ", text).strip())
    return f"{match['number']} {match['street']}" if match else None


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres; ample precision at a 50 m scale."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6_371_008.8 * np.arcsin(np.sqrt(a))


UNIT_KEY = ["operator", "charger_type", "charger_status"]
records = df[["charger_id", *UNIT_KEY, "source_feed", "latitude", "longitude",
              "station_address"]].copy()
records["charger_type"] = records["charger_type"].fillna("unknown")   # Upcoming sites
records["feed"] = records["source_feed"].fillna("(not recorded)")
records["street_key"] = records["station_address"].map(street_key)

# Candidate pairs share operator, type and status: a self-join on those keys.
pairs = records.merge(records, on=UNIT_KEY, suffixes=("_a", "_b"))
pairs = pairs[pairs["charger_id_a"] < pairs["charger_id_b"]].copy()
pairs["distance_m"] = haversine_m(pairs["latitude_a"], pairs["longitude_a"],
                                  pairs["latitude_b"], pairs["longitude_b"])
same_point = pairs["distance_m"] <= SAME_POINT_M
same_address = (pairs["street_key_a"].notna()
                & (pairs["street_key_a"] == pairs["street_key_b"])
                & (pairs["distance_m"] <= SAME_ADDRESS_RADIUS_M))
pairs["rule"] = np.select([same_point, same_address], ["same point", "same street address"],
                          default="")
duplicate_pairs = pairs[(pairs["feed_a"] != pairs["feed_b"]) & (pairs["rule"] != "")].copy()

# In this release every record is in at most one pair, so each pair simply names
# a representative. A future release that produces chains (a~b~c) stops here for
# review, rather than being resolved by a rule nobody has examined.
in_pairs = pd.concat([duplicate_pairs["charger_id_a"], duplicate_pairs["charger_id_b"]])
assert in_pairs.is_unique, "a record belongs to two duplicate pairs - review before flagging"

indexed = df.set_index("charger_id")
completeness = pd.DataFrame({
    "has_number": indexed["station_address"].str.match(r"\d", na=False),
    "populated": indexed.notna().sum(axis=1),
})


def representative(a, b):
    """The more complete of two records: street number, then filled fields, then lower id."""
    score = lambda cid: (completeness.at[cid, "has_number"], completeness.at[cid, "populated"], -cid)
    return a if score(a) >= score(b) else b


duplicate_pairs["duplicate_of"] = [
    representative(a, b) for a, b in zip(duplicate_pairs["charger_id_a"], duplicate_pairs["charger_id_b"])
]
duplicate_pairs["charger_id"] = np.where(
    duplicate_pairs["duplicate_of"] == duplicate_pairs["charger_id_a"],
    duplicate_pairs["charger_id_b"], duplicate_pairs["charger_id_a"],
)
probable_duplicates = (
    duplicate_pairs[["charger_id", "duplicate_of", "rule", "distance_m"]]
    .round({"distance_m": 1})
    .sort_values("charger_id")
    .reset_index(drop=True)
)
df["probable_duplicate_flag"] = df["charger_id"].isin(probable_duplicates["charger_id"])

record_step("probable duplicates", "same charger in two feeds: flagged, not merged",
            len(probable_duplicates))
display(
    probable_duplicates
    .assign(operator=lambda t: t["charger_id"].map(indexed["operator"]),
            type=lambda t: t["charger_id"].map(indexed["charger_type"]).fillna("Upcoming"),
            flagged_address=lambda t: t["charger_id"].map(indexed["station_address"]),
            kept_address=lambda t: t["duplicate_of"].map(indexed["station_address"]),
            flagged_feed=lambda t: t["charger_id"].map(indexed["source_feed"]),
            kept_feed=lambda t: t["duplicate_of"].map(indexed["source_feed"]))
    .sort_values(["rule", "distance_m"])
    .reset_index(drop=True)
)

  probable duplicates            23  same charger in two feeds: flagged, not merged


,charger_id,duplicate_of,rule,distance_m,operator,type,flagged_address,kept_address,flagged_feed,kept_feed
0,222,1302,same point,0.1,Exploren,AC,"179 Gillards Rd, Pokolbin, 2320",179 Gillards Rd Pokolbin NSW 2320 Australia,Existing Destination Chargers,Destination Charging R2
1,550,559,same point,0.1,Chargefox,AC,"43 Station St, Newcastle, 2293","44 Station St, Wickham NSW 2293, Australia",Existing Destination Chargers,Kerbside Charging R1
2,560,608,same point,0.1,Exploren,AC,"446 Dean St, Albury, 2640","520 David St, Albury NSW 2640, Australia",Existing Destination Chargers,Destination Charging R1
3,602,1580,same point,0.1,Exploren,AC,"51 Bathurst St, Condobolin, 2877",51 Bathurst St Condobolin NSW 2877 Australia,Existing Destination Chargers,Destination Charging R2
4,820,243,same point,0.1,Exploren,AC,"Buchanan Dr, South West Rocks NSW 2431, Australia","19 Buchanan Drive, South West Rocks, 2431",Destination Charging R1,Existing Destination Chargers
5,821,1691,same point,0.1,Chargefox,DC,"Bunnerong Rd, Sydney, 2036",801-899R Bunnerong Rd Chifley NSW 2036 Australia,Existing Fast Chargers,Kerbside Charging R1
6,909,1552,same point,0.1,Exploren,AC,"Merry Beach Rd, Kioloa, 2539",46 Merry Beach Rd Kioloa NSW 2539 Australia,Existing Destination Chargers,Destination Charging R2
7,940,1391,same point,0.1,Exploren,AC,"Princes Hwy, Ulladulla, 2539",222 Princes Hwy Ulladulla NSW 2539 Australia,Existing Destination Chargers,Destination Charging R2
8,967,1796,same point,0.1,Chargefox,AC,"Tathra Bermagui Rd, Tathra, 2550",1 Andy Poole Dr Tathra NSW 2550 Australia,Existing Destination Chargers,Destination Charging R2
9,1555,577,same point,0.1,Chargefox,DC,47-49A Cleary St Hamilton NSW 2303 Australia,"47-49A Cleary St, Newcastle, 2303",Kerbside Charging R1,Existing Fast Chargers


<a id="t2-ratingfields"></a>
## 10. Derive numeric rating fields

After reconciliation, the parser from Section 6 produces `power_kw_min`, `power_kw_max` and a tidy
component table.

Only compound ratings explicitly state a connector quantity. A plain value such as `22 kW` states
a power rating but does not provide a reliable connector total, so `connectors_in_rating` remains
missing for that format. This prevents the plug-count consistency check from treating an unstated
quantity as observed data.

`power_kw_max` supports simple charger-speed analysis, while `charger_power_ratings.csv` preserves
the parsed components of each rating, including the explicit quantities supplied by compound
values.

In [13]:
# The derived numeric fields are computed here, after reconciliation, so they
# always describe the rating string that actually survived the merge.
parsed = df["charger_rating_raw"].map(parse_rating)
df["rating_format"] = df["charger_rating_raw"].map(classify_rating)
df["power_kw_max"] = parsed.map(lambda comps: max(kw for _, kw in comps) if comps else np.nan)
df["power_kw_min"] = parsed.map(lambda comps: min(kw for _, kw in comps) if comps else np.nan)

# Only a compound rating ('2x350kW') states how many connectors exist. A plain
# '22 kW' states the power *per* connector and says nothing about their number,
# so treating it as one connector would invent a fact and make the plug-count
# cross-check below fire on almost every row.
df["connectors_in_rating"] = (
    parsed.map(lambda comps: sum(count for count, _ in comps) if comps else pd.NA)
          .where(df["rating_format"] == "compound", pd.NA)
          .astype("Int64")
)

record_step("rating parsed", "ratings resolved to a numeric kW value",
            int(df["power_kw_max"].notna().sum()))
record_step("rating unparsable", "placeholder ratings left as NA (e.g. 'AC')",
            int(df["power_kw_max"].isna().sum()))

# The compound ratings are also emitted as a tidy one-row-per-component table.
# `power_kw_max` answers "how fast is this charger?"; this answers "what is
# actually installed there?", which is what a normalised schema needs and what a
# single column cannot hold.
power_ratings = pd.DataFrame(
    [
        {"charger_id": charger_id, "component_no": position,
         "connector_count": count, "power_kw": kw}
        for charger_id, components in zip(df["charger_id"], parsed)
        for position, (count, kw) in enumerate(components, start=1)
    ],
    columns=["charger_id", "component_no", "connector_count", "power_kw"],
)
print(f"{len(power_ratings):,} rating components for "
      f"{power_ratings['charger_id'].nunique():,} chargers")
df["rating_format"].value_counts().to_frame("records")

  rating parsed               1,428  ratings resolved to a numeric kW value
  rating unparsable             518  placeholder ratings left as NA (e.g. 'AC')
1,527 rating components for 1,428 chargers


,records
rating_format,
number+unit,1314
non-numeric placeholder,518
compound,99
bare number,15


<a id="t2-consistency"></a>
## 11. Check cross-field consistency

This section checks whether related fields within each retained record agree. Three conditions are
flagged:

- a record classified as `DC` whose raw rating contains the `AC` placeholder;
- a compound rating whose parsed connector total differs from `number_of_plugs`; and
- a record with no `station_name`.

The checks do not overwrite or remove data. They create boolean flags so contradictions and missing
names remain visible in the validation summary and can be considered during external matching.
Plain ratings are excluded from the connector-total comparison because they do not explicitly state
the number of installed connectors.

In [14]:
# Cross-field checks. These change no values; they mark records whose own fields
# contradict each other, so consistency can be quantified and Task 3 can
# avoid augmenting a record that is internally unreliable.
df["type_rating_conflict_flag"] = (
    (df["charger_type"] == "DC")
    & df["charger_rating_raw"].str.upper().eq("AC").fillna(False)
)
df["plug_count_conflict_flag"] = (
    df["connectors_in_rating"].notna()
    & df["number_of_plugs"].notna()
    & (df["connectors_in_rating"] != df["number_of_plugs"])
)
df["missing_name_flag"] = df["station_name"].isna()

record_step("type vs rating", "DC records whose rating reads 'AC'",
            int(df["type_rating_conflict_flag"].sum()))
record_step("plugs vs connectors", "plug count disagrees with the parsed rating",
            int(df["plug_count_conflict_flag"].sum()))
record_step("station name missing", "records with no station name",
            int(df["missing_name_flag"].sum()))

  type vs rating                  0  DC records whose rating reads 'AC'
  plugs vs connectors            36  plug count disagrees with the parsed rating
  station name missing        1,430  records with no station name


<a id="t2-spatial"></a>
## 12. Integrate ASGS SA4 geography

The cleaned charger points are joined to Australian Bureau of Statistics Statistical Area Level 4
(SA4) boundaries. This adds a consistent regional classification for geographic analysis and avoids
relying on the placeholder localities found in some source addresses.

### 12.1 Reconcile coordinate reference systems

The two datasets use different coordinate reference systems: the ABS boundaries use GDA2020
(`EPSG:7844`), while the charger coordinates use WGS84 (`EPSG:4326`). Although the difference is
small at the current epoch, the polygons are transformed explicitly so both datasets use the same
reference system before the spatial join.

`always_xy := true` is a critical implementation detail. `ST_Point` uses `(x, y)`, or longitude
followed by latitude, while EPSG axis definitions may use a different formal order. The option keeps
the transformed polygons consistent with the charger-point coordinate order.

In [15]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# The boundary file is GDA2020 (EPSG:7844); the TfNSW coordinates are WGS84
# (EPSG:4326). The two are close - under about 2 m at the current epoch - but
# "close enough" is not a decision worth leaving implicit in a spatial join, so
# the polygons are reprojected explicitly.
#
# `always_xy := true` is the part that matters. EPSG:4326 formally orders its
# axes latitude-then-longitude, so without it the transform returns coordinates
# in the opposite order from ST_Point(longitude, latitude) and every point lands
# in the ocean off East Africa. If the local PROJ build cannot perform the
# transform, the untransformed geometry is used and the fallback is logged: a
# sub-2 m offset cannot move a charger across an SA4 boundary kilometres wide.
TRANSFORMED = f"ST_Transform(geom, '{SOURCE_CRS}', '{TARGET_CRS}', always_xy := true)"
SA4_COLUMNS = """
    SA4_CODE26 AS sa4_code,
    SA4_NAME26 AS sa4_name,
    GCC_CODE26 AS gcc_code,
    GCC_NAME26 AS gcc_name,
    STE_CODE26 AS state_code,
    STE_NAME26 AS state_name,
    AREASQKM26 AS area_sqkm
"""
try:
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, {TRANSFORMED} AS geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"reprojected {SOURCE_CRS} -> {TARGET_CRS} (always_xy)"
except duckdb.Error as error:
    print(f"WARNING: ST_Transform unavailable ({str(error)[:120]}); "
          "using GDA2020 coordinates directly (offset < 2 m).")
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"{SOURCE_CRS} treated as {TARGET_CRS} (offset < 2 m)"

record_step("CRS reconciled", crs_note,
            int(con.execute("SELECT COUNT(*) FROM sa4").fetchone()[0]))
con.execute("SELECT state_name, COUNT(*) AS regions FROM sa4 GROUP BY 1 ORDER BY 2 DESC").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  CRS reconciled                108  reprojected EPSG:7844 -> EPSG:4326 (always_xy)


,state_name,regions
0,New South Wales,30
1,Queensland,21
2,Victoria,19
3,Western Australia,12
4,South Australia,9
5,Tasmania,6
6,Northern Territory,4
7,Australian Capital Territory,3
8,Other Territories,3
9,Outside Australia,1


### 12.2 Assign points to SA4 polygons

Each charger is converted to a point and tested for containment in the Australian SA4 polygons.
The join initially uses all 108 Australian SA4 records rather than filtering to NSW. This allows an
interstate charger to be identified as a state error instead of appearing indistinguishable from a
failed spatial match. Final validation confirms that every assigned charger belongs to NSW.

A `LEFT JOIN` preserves unmatched chargers as rows with null region fields. An inner join would
silently remove them and conceal a spatial or coordinate problem.

In [16]:
# The cleaned chargers are handed to DuckDB as a view over the DataFrame - no
# copy, no intermediate file.
chargers_for_join = df[["charger_id", "longitude", "latitude"]].astype(
    {"charger_id": "int64", "longitude": "float64", "latitude": "float64"}
)
con.register("chargers_py", chargers_for_join)

# ST_Point takes (x, y) = (longitude, latitude). Passing them the other way round
# is the most common error in this kind of join and it fails silently - every
# point simply matches nothing.
#
# The join runs against all 108 Australian SA4s rather than a pre-filtered NSW
# subset. Filtering first would hide a genuine problem: a charger whose
# coordinates put it outside NSW would come back unmatched and look like a
# geometry failure, instead of being correctly reported as an interstate point.
con.execute("""
    CREATE OR REPLACE TABLE charger_sa4 AS
    SELECT c.charger_id,
           s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name, s.state_name,
           CASE WHEN s.sa4_code IS NULL THEN NULL ELSE 'point-in-polygon' END
               AS sa4_match_method
    FROM chargers_py AS c
    LEFT JOIN sa4 AS s
      ON ST_Contains(s.geom, ST_Point(c.longitude, c.latitude))
""")

matched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NOT NULL"
).fetchone()[0]
print(f"{matched:,} of {len(df):,} chargers matched by point-in-polygon "
      f"({matched / len(df) * 100:.1f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1,945 of 1,946 chargers matched by point-in-polygon (99.9%)


### 12.3 Handle points outside polygon boundaries

Coastal infrastructure can lie a few metres outside the digitised ABS coastline, causing a valid
charger point to miss every polygon. Such records are assigned to the nearest SA4 only when the
polygon lies within the stated tolerance of approximately 2 km. The value of `sa4_match_method`
records whether assignment used exact containment or the nearest-polygon fallback.

The tolerance is expressed as 0.02 degrees because this join uses geographic coordinates and
`ST_Distance` therefore returns degrees. At NSW latitudes this is approximately 2 km and serves as a
conservative sanity threshold rather than a precise travel distance.

In [17]:
# A charger on a jetty, a reclaimed wharf or a coastal car park can sit a few
# metres outside the coastline the ABS digitised, and point-in-polygon returns
# nothing for it. Rather than dropping those records or leaving the field null,
# each is assigned the nearest SA4 - but only within a stated tolerance, and the
# method is recorded in `sa4_match_method` so exact matches can be told
# apart from assisted ones.
NEAREST_TOLERANCE_DEGREES = 0.02       # ~2 km at NSW latitudes

unmatched_count = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]

if unmatched_count:
    con.execute(f"""
        CREATE OR REPLACE TABLE charger_sa4_nearest AS
        WITH candidates AS (
            SELECT u.charger_id, s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name,
                   s.state_name,
                   ST_Distance(s.geom, ST_Point(u.longitude, u.latitude)) AS distance_deg,
                   ROW_NUMBER() OVER (
                       PARTITION BY u.charger_id
                       ORDER BY ST_Distance(s.geom, ST_Point(u.longitude, u.latitude))
                   ) AS rank
            FROM chargers_py AS u
            JOIN charger_sa4 AS m
              ON m.charger_id = u.charger_id AND m.sa4_code IS NULL
            CROSS JOIN sa4 AS s
            WHERE NOT ST_IsEmpty(s.geom)
        )
        SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
               state_name, distance_deg
        FROM candidates
        WHERE rank = 1 AND distance_deg <= {NEAREST_TOLERANCE_DEGREES}
    """)
    con.execute("""
        UPDATE charger_sa4 AS t
        SET sa4_code = n.sa4_code, sa4_name = n.sa4_name,
            gcc_code = n.gcc_code, gcc_name = n.gcc_name,
            state_name = n.state_name, sa4_match_method = 'nearest-polygon'
        FROM charger_sa4_nearest AS n
        WHERE t.charger_id = n.charger_id
    """)
    rescued = con.execute(
        "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_match_method = 'nearest-polygon'"
    ).fetchone()[0]
    print(f"{rescued} of {unmatched_count} unmatched charger(s) assigned by nearest polygon")
else:
    print("Every charger fell inside an SA4 polygon; no fallback needed.")

still_unmatched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]
record_step("SA4 assigned", "chargers given an SA4 region", len(df) - still_unmatched)
record_step("SA4 unassigned", "chargers left without an SA4 region", still_unmatched)

1 of 1 unmatched charger(s) assigned by nearest polygon
  SA4 assigned                1,946  chargers given an SA4 region
  SA4 unassigned                  0  chargers left without an SA4 region


### 12.4 Merge regional attributes

The SA4 result is joined to the cleaned DataFrame on `charger_id` with
`validate="one_to_one"`. Pandas raises an error if either side contains a duplicated key, preventing
the join from silently increasing the number of charger records.

Two NSW codes represent non-spatial ABS categories: `Migratory - Offshore - Shipping (NSW)` and
`No usual address (NSW)`. They have no polygon and therefore cannot contain a charger point. They
remain in the SA4 reference output with `has_boundary = False`, which distinguishes them from
spatial regions that happen to contain no chargers.

The separate NSW SA4 output also allows coverage summaries to include regions with zero chargers;
a table produced only from matched chargers cannot show an absent region.

In [18]:
sa4_lookup = con.execute("""
    SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
           state_name, sa4_match_method
    FROM charger_sa4
""").df()

df = df.merge(sa4_lookup, on="charger_id", how="left", validate="one_to_one")

# The NSW SA4 reference table becomes its own output: Task 4 stores it as the
# region dimension, and it is what makes a region with *no* chargers visible.
# A join result alone can only ever show regions that already have one.
sa4_regions_nsw = con.execute("""
    SELECT sa4_code, sa4_name, gcc_code, gcc_name, state_name,
           ROUND(area_sqkm, 1) AS area_sqkm,
           geom IS NOT NULL AS has_boundary    -- FALSE for the ABS non-spatial codes
    FROM sa4
    WHERE state_name = 'New South Wales'
    ORDER BY sa4_code
""").df()

with_boundary = sa4_regions_nsw["has_boundary"]
print(f"{len(sa4_regions_nsw)} NSW SA4 codes: {with_boundary.sum()} regions with a boundary and "
      f"{(~with_boundary).sum()} ABS non-spatial codes "
      f"({', '.join(sa4_regions_nsw.loc[~with_boundary, 'sa4_name'])})")
print(f"{df['sa4_code'].nunique()} of the {with_boundary.sum()} regions contain at least one charger")
df[["charger_id", "operator", "charger_type", "latitude", "longitude",
    "sa4_code", "sa4_name", "sa4_match_method"]].head()

30 NSW SA4 codes: 28 regions with a boundary and 2 ABS non-spatial codes (Migratory - Offshore - Shipping (NSW), No usual address (NSW))
28 of the 28 regions contain at least one charger


,charger_id,operator,charger_type,latitude,longitude,sa4_code,sa4_name,sa4_match_method
0,1,EVUp,AC,-32.262242,150.890139,106,Hunter Valley exc Newcastle,point-in-polygon
1,2,BP Australia,DC,-33.811004,150.849597,116,Sydney - Blacktown,point-in-polygon
2,3,NRMA,DC,-30.511874,151.669395,110,New England and North West,point-in-polygon
3,4,Chargefox,AC,-33.774101,151.167035,121,Sydney - North Sydney and Hornsby,point-in-polygon
4,5,Tesla,AC,-28.641819,153.613633,112,Richmond - Tweed,point-in-polygon


<a id="t2-validate"></a>
## 13. Validate the cleaned data

The final checks encode assumptions required by later processing: `charger_id` must be unique,
every non-null `charger_type` must be AC or DC, each power-rating row must reference an existing
charger, coordinates must be valid, and every assigned region must be in NSW. Expressing these
assumptions as assertions allows a changed source release to stop at the point of failure rather
than produce an incorrect downstream result.

In [19]:
# Post-conditions. Each is an assumption the rest of the pipeline relies on, so
# each is asserted rather than eyeballed: a future TfNSW release that breaks one
# should stop the notebook here, not surface as a wrong number in Task 4.
problems = []

if df["charger_id"].duplicated().any():
    problems.append("charger_id is not unique")
if df["charger_id"].isna().any():
    problems.append("charger_id contains nulls")
if not set(df["charger_type"].dropna()) <= {"AC", "DC"}:
    problems.append(f"unexpected charger_type values: {set(df['charger_type'].dropna())}")
if not set(df["charger_status"].dropna()) <= {"Operational", "Upcoming", "Unknown"}:
    problems.append("unexpected charger_status values")
if not bool(df["latitude"].dropna().between(*NSW_LAT_RANGE).all()):
    problems.append("latitude outside the NSW range")
if not bool(power_ratings["charger_id"].isin(df["charger_id"]).all()):
    problems.append("power_ratings references an unknown charger_id")

duplicate_targets = probable_duplicates["duplicate_of"]
if not duplicate_targets.isin(df["charger_id"]).all():
    problems.append("a probable duplicate points to an unknown charger_id")
if duplicate_targets.isin(probable_duplicates["charger_id"]).any():
    problems.append("a representative record is itself flagged as a duplicate")

off_state = df.loc[df["state_name"].notna() & (df["state_name"] != "New South Wales")]
if len(off_state):
    problems.append(f"{len(off_state)} charger(s) matched an SA4 outside NSW")

print("Validation:", "PASSED" if not problems else "FAILED")
for problem in problems:
    print("  -", problem)

summary = pd.DataFrame({
    "metric": [
        "raw records", "clean records", "distinct sites", "distinct operators",
        "DC chargers", "AC chargers", "upcoming sites",
        "probable duplicates (flagged, not merged)",
        "records with an SA4", "NSW SA4 regions with a boundary",
        "NSW SA4 regions with no charger", "ABS non-spatial SA4 codes (no boundary)",
    ],
    "value": [
        len(raw), len(df), df["site_id"].nunique(), df["operator"].nunique(),
        int((df["charger_type"] == "DC").sum()), int((df["charger_type"] == "AC").sum()),
        int((df["charger_status"] == "Upcoming").sum()),
        int(df["probable_duplicate_flag"].sum()),
        int(df["sa4_code"].notna().sum()), int(with_boundary.sum()),
        int((~sa4_regions_nsw.loc[with_boundary, "sa4_code"].isin(df["sa4_code"])).sum()),
        int((~with_boundary).sum()),
    ],
})
summary

Validation: PASSED


,metric,value
0,raw records,1958
1,clean records,1946
2,distinct sites,1935
3,distinct operators,42
4,DC chargers,431
5,AC chargers,1417
6,upcoming sites,98
7,"probable duplicates (flagged, not merged)",23
8,records with an SA4,1946
9,NSW SA4 regions with a boundary,28


The regional distribution is calculated from the full list of spatial NSW SA4 regions rather than
only the charger join. A region without chargers would therefore appear with a count of zero instead
of disappearing from the result. Non-spatial ABS codes are excluded because they do not represent
geographic areas.

In [20]:
chargers_by_sa4 = (
    sa4_regions_nsw.loc[sa4_regions_nsw["has_boundary"], ["sa4_code", "sa4_name", "gcc_name"]]
    .merge(
        df.groupby("sa4_code")
          .agg(chargers=("charger_id", "count"),
               dc_chargers=("charger_type", lambda s: int((s == "DC").sum())),
               plugs=("number_of_plugs", "sum"))
          .reset_index(),
        on="sa4_code", how="left",
    )
    .fillna({"chargers": 0, "dc_chargers": 0, "plugs": 0})
    .astype({"chargers": int, "dc_chargers": int, "plugs": int})
    .sort_values("chargers", ascending=False)
    .reset_index(drop=True)
)
chargers_by_sa4

,sa4_code,sa4_name,gcc_name,chargers,dc_chargers,plugs
0,118,Sydney - Eastern Suburbs,Greater Sydney,219,34,399
1,117,Sydney - City and Inner South,Greater Sydney,139,19,388
2,106,Hunter Valley exc Newcastle,Rest of NSW,127,9,361
3,101,Capital Region,Rest of NSW,117,27,411
4,103,Central West,Rest of NSW,113,17,269
5,120,Sydney - Inner West,Greater Sydney,113,17,238
6,121,Sydney - North Sydney and Hornsby,Greater Sydney,108,41,346
7,111,Newcastle and Lake Macquarie,Rest of NSW,86,13,250
8,114,Southern Highlands and Shoalhaven,Rest of NSW,76,9,203
9,112,Richmond - Tweed,Rest of NSW,76,14,213


<a id="t2-outputs"></a>
## 14. Save the outputs

Four datasets and the cleaning ledger are written to `data/interim/`. The cleaned charger and
power-rating outputs support subsequent augmentation and database construction, while the probable
duplicate and SA4 outputs retain review evidence and geographic reference information.

The cleaned charger file keeps selected original values, including `operator_raw`,
`charger_rating_raw` and `station_address_raw`, alongside their cleaned forms. This makes each
transformation reviewable without requiring the acquisition and cleaning stages to be rerun.

In [21]:
COLUMN_ORDER = [
    "charger_id", "site_id",
    "station_name", "station_address", "lga_name", "postcode",
    "latitude", "longitude",
    "sa4_code", "sa4_name", "gcc_code", "gcc_name", "state_name", "sa4_match_method",
    "operator", "charger_type", "charger_status",
    "number_of_plugs", "power_kw_min", "power_kw_max", "connectors_in_rating",
    "rating_format", "charger_rating_raw", "operator_raw", "charger_type_raw",
    "source_feed", "source_object_id", "postcode_reported", "station_address_raw",
    "operator_truncated_flag", "postcode_conflict_flag", "postcode_valid_flag",
    "locality_placeholder_flag", "coordinate_valid_flag",
    "type_rating_conflict_flag", "plug_count_conflict_flag", "missing_name_flag",
    "probable_duplicate_flag",
]
chargers_clean = df[[column for column in COLUMN_ORDER if column in df.columns]].copy()

chargers_clean.to_csv(CLEAN_CHARGERS_PATH, index=False)
power_ratings.to_csv(RATINGS_PATH, index=False)
probable_duplicates.to_csv(DUPLICATES_PATH, index=False)
sa4_regions_nsw.to_csv(SA4_NSW_PATH, index=False)
CLEANING_LOG_PATH.write_text(
    json.dumps(
        {"raw_records": len(raw), "clean_records": len(chargers_clean),
         "steps": cleaning_log},
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

for path in (CLEAN_CHARGERS_PATH, RATINGS_PATH, DUPLICATES_PATH, SA4_NSW_PATH, CLEANING_LOG_PATH):
    print(f"  {path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size / 1024:,.1f} KB)")

con.close()
pd.DataFrame(cleaning_log)

  data\interim\ev_chargers_clean.csv  (708.7 KB)
  data\interim\charger_power_ratings.csv  (21.5 KB)
  data\interim\probable_duplicates.csv  (0.7 KB)
  data\interim\sa4_regions_nsw.csv  (2.3 KB)
  data\interim\cleaning_log.json  (3.2 KB)


,step,detail,records_affected
0,whitespace normalised,cells whose text was rewritten,808
1,empty -> NA,empty strings converted to missing values,3638
2,operator canonicalised,values rewritten to a canonical name,181
3,operator truncated,unresolved truncations flagged for review,19
4,type/status separated,'Upcoming' moved into charger_status,98
5,addresses normalised,addresses reduced to one clean line,1958
6,postcode recovered,postcode recovered from the address text,120
7,postcode conflict,PCODE disagrees with the address postcode,35
8,postcode invalid,postcode outside the NSW ranges,3
9,locality placeholder,'Sydney' used in place of the real suburb,399
